# 🐸 Wednesday Frog – Flow Matching Training (Kaggle / Colab)

**Kaggle:** Settings → Accelerator = *GPU T4 x2* oder *P100*, Internet = *On*. Unter *Add Input* dein Dataset `wednesday-frogs` (die Zip aus `scripts/prepare.py`) hinzufügen.  
**Colab:** Runtime → Change runtime type → GPU. Zip in Google Drive legen.

Bei Timeout einfach alles nochmal ausführen – `train.py` setzt automatisch bei `last.pt` fort (sofern das Output-Verzeichnis erhalten bleibt, z.B. auf Drive).

In [ ]:
REPO = "https://github.com/alexanderulbrich/wednesday.git"
!git clone -q $REPO wednesday || (cd wednesday && git pull -q)
%cd wednesday
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if os.path.exists("/kaggle"):
    DATA = "/kaggle/input/wednesday-frogs"          # Kaggle entpackt Datasets automatisch
    OUT  = "/kaggle/working/run64"
else:  # Colab
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/data && unzip -qo /content/drive/MyDrive/wednesday-frogs.zip -d /content/data
    DATA = "/content/data"
    OUT  = "/content/drive/MyDrive/wednesday-runs/run64"   # auf Drive -> überlebt Timeouts
RES = 64
!ls $DATA/$RES | head -3 && ls $DATA/$RES | wc -l

In [ ]:
!python train.py --data $DATA/$RES --res $RES --out $OUT --steps 80000 --batch 64 --sample-every 2000 --save-every 5000 --workers 2

## Fortschritt anschauen

In [ ]:
from IPython.display import Image, display
import glob
samples = sorted(glob.glob(f"{OUT}/samples/*.png"))
for s in samples[-3:]:
    print(s); display(Image(s))

Am Ende `frog.pt` (nur EMA-Gewichte, klein) aus dem Output herunterladen und auf dem Mac `python generate.py --ckpt frog.pt` ausführen.

In [ ]:
# Zwischenstand als frog.pt exportieren (falls Training noch nicht fertig ist)
import torch
ck = torch.load(f"{OUT}/last.pt", map_location="cpu", weights_only=False)
torch.save({"ema": ck["ema"], "model_config": ck["model_config"], "step": ck["step"]}, f"{OUT}/frog.pt")
print("step", ck["step"], "->", f"{OUT}/frog.pt")